# Étape 1 : Exploration et Filtrage de The Stack Dedup

Dans ce notebook, nous explorons le corpus original [bigcode/the-stack-dedup](https://huggingface.co/datasets/bigcode/the-stack-dedup) afin d'isoler les scripts Python pertinents pour notre cas d'usage. 
L'objectif est de créer un sous-ensemble d'entraînement focalisé exclusivement sur la manipulation de données avec `pandas` et `matplotlib`.

In [ ]:
import re
from datasets import load_dataset

# Chargement d'un échantillon pour l'exploration
print("Chargement du dataset The Stack (split train)...")
dataset = load_dataset(
    "bigcode/the-stack-dedup", 
    split="train", 
    data_dir="data/python",
    streaming=True, # Mode streaming pour ne pas surcharger la RAM en exploration
    trust_remote_code=True
)

# Aperçu du premier élément
sample = next(iter(dataset))
print(f"Taille du code source complet : {len(sample['content'])} caractères")

## Stratégie de Filtrage par Expression Régulière
Nous appliquons un filtre strict pour ne conserver que les fichiers déclarant explicitement l'importation de nos bibliothèques cibles.

In [ ]:
pattern = re.compile(r"(import\s+pandas|import\s+matplotlib)")

def has_target_libraries(text):
    return bool(pattern.search(text))

# Test sur quelques exemples du flux
matched_examples = []
for example in dataset:
    if has_target_libraries(example["content"]):
        matched_examples.append(example)
    if len(matched_examples) == 3:
        break

print("Exemple de script conservé (extrait) :")
print("-" * 40)
print(matched_examples[0]["content"][:300] + "\n...")